## INCREMENTAL DATA INGESTION

In [0]:
%python

import json

dbutils.widgets.text("sources", "")
sources_json = dbutils.widgets.get("sources")

# Convert the JSON list back into a Python list
sources = json.loads(sources_json)
print(sources)


In [0]:
%python

for file_name in sources:

    source_path = f"/Volumes/flights_project/raw/raw_volume/raw_data/{file_name}/"
    schema_location = f"/Volumes/flights_project/bronze/bronze_volume/{file_name}/schema"

    # readStream: Defines how to read
    df = (
        spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", "csv")
                .option("cloudFiles.schemaLocation", schema_location)
                .option("cloudFiles.schemaEvolutionMode", "rescue")         # preserves that data in _rescued_data instead of evolving the schema automatically
                .option("header", True)
                .load(source_path)
    )

    target_path = f"/Volumes/flights_project/bronze/bronze_volume/{file_name}/data"
    checkpoint_location = f"/Volumes/flights_project/bronze/bronze_volume/{file_name}/checkpoint"

    # writeStream.start(): Starts execution.
    query = (df.writeStream
        .format("delta")
        .outputMode("append")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_location)  
        .option("path", target_path)
        .start()
    )

    query.awaitTermination()            # wait until the streaming query finishes or fails

